# Pre-M0.3 — IQ Indexing and Slicing

## Unit Objective

Master **NumPy indexing and slicing** on IQ signal tensors. You will learn how integer indexing can remove an axis and how slicing preserves it, and you will apply these operations to extract and manipulate IQ data along all three axes of the canonical `(N, 2, L)` layout.

## What You Will Learn

1. How the IQ tensor is structured: `X.shape == (N, 2, L)` — axis 0 = examples, axis 1 = I/Q components (0=I, 1=Q), axis 2 = time samples.
2. The difference between **integer indexing** (removes an axis) and **slicing** (preserves an axis).
3. How to extract a single example, a single component, a range of samples, or a subsampled signal.
4. How to diagnose an axis swap error and correct it.
5. How to verify that two independent power computations agree.

## IQ Data Used in This Notebook

All exercises use a synthetic IQ dataset with:
- **N** = 50 examples (independent signals)
- **2** components: I (in-phase) and Q (quadrature)
- **L** = 200 time samples per example
- dtype = `np.float32`
- Random seed = 42 for reproducibility

We construct the data with a known ground truth so that later checks are verifiable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

N = 50   # number of examples
L = 200  # time samples per example

# Build IQ tensor: shape (N, 2, L), dtype float32
# axis 0 = examples, axis 1 = I/Q (0=I, 1=Q), axis 2 = time
X = rng.standard_normal((N, 2, L)).astype(np.float32)

print(f"X.shape  = {X.shape}")
print(f"X.dtype  = {X.dtype}")
print(f"X.ndim   = {X.ndim}")
assert X.shape == (N, 2, L), f"Expected ({N}, 2, {L}), got {X.shape}"
assert X.dtype == np.float32
print("IQ tensor created successfully.")

---
## Guided Development: Indexing vs. Slicing

NumPy gives us two fundamentally different ways to access elements of an array:

| Operation | Example | Effect on shape |
|-----------|---------|------------------|
| **Integer index** on axis `i` | `X[0]` | Axis `i` is **removed** |
| **Slice** on axis `i` | `X[0:1]` | Axis `i` is **preserved** |

This distinction is critical. Removing an axis changes the semantics of downstream operations. Preserving it keeps the batch dimension intact.

### Rule 1 — Integer indexing removes an axis

When you index a single position along an axis with an integer, that axis collapses and disappears from the result.

In [ ]:
# X[0] — integer index on axis 0 → removes axis 0
# Result shape: (2, L) — we get the first example as a 2D array
single_example = X[0]
print(f"X[0].shape        = {single_example.shape}")   # (2, 200)
print(f"X.shape           = {X.shape}")                 # (50, 2, 200)
print(f"Axis 0 removed:   {X.ndim}D → {single_example.ndim}D")

# X[0, 0, :] — three integer indices + slice → only the slice axis remains
single_i_example = X[0, 0, :]
print(f"X[0,0,:].shape    = {single_i_example.shape}") # (200,)
print(f"Two axes removed: {X.ndim}D → {single_i_example.ndim}D")

### Rule 2 — Slicing preserves an axis

When you use a slice (`start:stop` or `start:stop:step`), even if the slice contains only one element, the axis remains.

In [ ]:
# X[0:1] — slice on axis 0 → axis 0 is preserved
batch_of_one = X[0:1]
print(f"X[0:1].shape      = {batch_of_one.shape}")   # (1, 2, 200)
print(f"X.shape           = {X.shape}")                # (50, 2, 200)
print(f"Axis 0 preserved: {X.ndim}D → {batch_of_one.ndim}D")

# Why this matters: downstream ops keep the batch dimension
print(f"\nX[0].mean()      = {X[0].mean():.6f}    (scalar-ish: shape {X[0].mean().shape})")
print(f"X[0:1].mean()    = {X[0:1].mean(axis=(1,2)).shape}  (still a batch: shape {X[0:1].mean(axis=(1,2)).shape})")

---
## Common Indexing Patterns for IQ Data

Let us walk through the most useful indexing patterns on our tensor `X` of shape `(N, 2, L)`.

### Pattern 1 — Select one example: `X[0]`
Integer index on axis 0 → removes the batch axis.

In [ ]:
one = X[0]
print(f"X[0].shape = {one.shape}")   # (2, 200)
print(f"I component: X[0,0,:].shape = {one[0,:].shape}")
print(f"Q component: X[0,1,:].shape = {one[1,:].shape}")

### Pattern 2 — Select one example (preserve batch): `X[0:1]`
Slice on axis 0 → preserves the batch axis.

In [ ]:
one_batch = X[0:1]
print(f"X[0:1].shape = {one_batch.shape}")  # (1, 2, 200)
print(f"Batch axis preserved: ndim = {one_batch.ndim}")

### Pattern 3 — Select I across all examples: `X[:, 0, :]`
Slice on axis 0 (all), integer on axis 1 (removes I/Q axis), slice on axis 2 (all).

In [ ]:
I_all = X[:, 0, :]    # all examples, I component, all time samples
print(f"X[:,0,:].shape = {I_all.shape}")   # (50, 200)
print(f"Axis 1 removed: {X.ndim}D → {I_all.ndim}D")

### Pattern 4 — Select Q across all examples: `X[:, 1, :]`
Same pattern but extracting the quadrature component.

In [ ]:
Q_all = X[:, 1, :]
print(f"X[:,1,:].shape = {Q_all.shape}")   # (50, 200)

### Pattern 5 — Select I of one example: `X[0, 0, :]`
Three integer/slice indices → removes axes 0 and 1.

In [ ]:
i_single = X[0, 0, :]
print(f"X[0,0,:].shape = {i_single.shape}")   # (200,)
print(f"1D array: ndim = {i_single.ndim}")

### Pattern 6 — Extract a sub-range of samples: `X[0, :, 10:30]`
Slice on axis 2 to grab samples 10 through 29.

In [ ]:
sub_range = X[0, :, 10:30]
print(f"X[0,:,10:30].shape = {sub_range.shape}")  # (2, 20)
print(f"I sub-range: {sub_range[0,:].shape}")
print(f"Q sub-range: {sub_range[1,:].shape}")

### Pattern 7 — Subsample by stride: `X[:, :, ::2]`
Slice with step 2 on axis 2 → take every other sample.

In [ ]:
subsampled = X[:, :, ::2]
print(f"X[:,:,::2].shape = {subsampled.shape}")  # (50, 2, 100)
print(f"Original samples: {X.shape[2]}, Subsampled: {subsampled.shape[2]}")

### Shape Summary Table

| Operation | Result shape | Axes removed |
|-----------|-------------|--------------|
| `X[0]` | `(2, 200)` | axis 0 |
| `X[0:1]` | `(1, 2, 200)` | none |
| `X[:, 0, :]` | `(50, 200)` | axis 1 |
| `X[:, 1, :]` | `(50, 200)` | axis 1 |
| `X[0, 0, :]` | `(200,)` | axes 0, 1 |
| `X[0, :, 10:30]` | `(2, 20)` | axis 0 |
| `X[:, :, ::2]` | `(50, 2, 100)` | none |

---
## Small Examples

Let us visualize a few slices to build intuition.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))

# Example 0, I component, first 50 samples
axes[0].plot(X[0, 0, :50])
axes[0].set_title("X[0, 0, :50] — example 0, I")
axes[0].set_xlabel("Sample index")

# Example 0, Q component, first 50 samples
axes[1].plot(X[0, 1, :50], color="C1")
axes[1].set_title("X[0, 1, :50] — example 0, Q")
axes[1].set_xlabel("Sample index")

# Subsampled: every 4th sample
axes[2].plot(X[0, 0, ::4])
axes[2].set_title("X[0, 0, ::4] — subsampled I")
axes[2].set_xlabel("Sample index (every 4th)")

plt.tight_layout()
plt.show()

---
## Student Exercises

Complete each exercise by writing the correct NumPy expression. Run the cell and check the printed shape.

### Exercise 1 — Select example 5
**Task:** Write an expression that selects example index 5 from `X`. What should the resulting shape be?

In [ ]:
# STUDENT: write your expression here
# exercise_1 = ...
# print(f"exercise_1.shape = {exercise_1.shape}")

### Exercise 2 — Select I component across all examples
**Task:** Extract only the I component (index 0 on axis 1) for every example. What shape do you expect?

In [ ]:
# STUDENT: write your expression here
# exercise_2 = ...
# print(f"exercise_2.shape = {exercise_2.shape}")

### Exercise 3 — Select Q component of example 10
**Task:** Extract the Q component of example 10. What shape do you expect?

In [ ]:
# STUDENT: write your expression here
# exercise_3 = ...
# print(f"exercise_3.shape = {exercise_3.shape}")

### Exercise 4 — Extract samples 20 through 39
**Task:** From example 3, extract only the I and Q components for sample indices 20 to 39 (inclusive). What shape do you expect?

In [ ]:
# STUDENT: write your expression here
# exercise_4 = ...
# print(f"exercise_4.shape = {exercise_4.shape}")

### Exercise 5 — Preserve the batch dimension
**Task:** Select example 7, but keep the batch axis (result should have 3 dimensions). What is the resulting shape?

In [ ]:
# STUDENT: write your expression here
# exercise_5 = ...
# print(f"exercise_5.shape = {exercise_5.shape}")

### Exercise 6 — Predict the output shape
**Task:** Before running the code, predict the shape of `X[1:4, 0, 50:100]`. Then run the cell to verify.

In [ ]:
# PREDICT: what is the shape? Write your answer as a comment.
# exercise_6 = X[1:4, 0, 50:100]
# print(f"exercise_6.shape = {exercise_6.shape}")

---
## Pass Criterion Challenges

Three independent gates. All must pass for **PRE-M0.3 FINAL STATUS: PASS**.

---
### PC-1 — Independent Axis Explanation

You must answer the 6 questions below **without looking at the solutions**. Write your answers in the Markdown cell provided.

#### STUDENT AXIS EXPLANATION

Answer each question in the space below.

**Q1.** What is `X.shape` and what does each axis represent?

YOUR ANSWER:

**Q2.** What is the shape of `X[0]`? Which axis was removed and why?

YOUR ANSWER:

**Q3.** What is the shape of `X[0:1]`? Why is it different from `X[0]`?

YOUR ANSWER:

**Q4.** What is the shape of `X[:, 0, :]`? Which axis was removed?

YOUR ANSWER:

**Q5.** What is the shape of `X[0, 0, :]`? How many axes were removed?

YOUR ANSWER:

**Q6.** If you have a result of shape `(50, 200)` from `X[:, 0, :]`, can you recover the original `(50, 2, 200)` shape? Explain why or why not in terms of lost information.

YOUR ANSWER:

In [ ]:
# Student/instructor: change to True after verifying the answers above.
AXES_EXPLANATION_VERIFIED = False  # Set to True manually after verification

print(f"AXES_EXPLANATION_VERIFIED = {AXES_EXPLANATION_VERIFIED}")
if AXES_EXPLANATION_VERIFIED:
    print("PC-1: PASS")
else:
    print("PC-1: WAIT — complete the axis explanation above")

---
### PC-2 — Injected Axis Swap

A bug was injected into the data pipeline. The I/Q axis and the time axis were transposed. Your job is to **diagnose** and **fix** the swap.

#### The injected error

The following code simulates a common mistake: someone transposed axes 1 and 2, swapping the I/Q axis with the time axis.

In [ ]:
# INJECTED ERROR — do NOT modify this cell
X_swapped = np.transpose(X, (0, 2, 1))

print(f"X.shape      = {X.shape}")
print(f"X_swapped.shape = {X_swapped.shape}")
print(f"\nSomething is wrong: the I/Q axis (size 2) and time axis (size {L}) were swapped.")

#### STUDENT ATTEMPT

Inspect `X_swapped` and answer:
1. What is the shape of `X_swapped`?
2. Which axis now holds the I/Q components and which holds time samples?
3. Why is this shape wrong for our IQ contract?
4. Write the correction to produce `X_fixed` that matches `X` exactly.

Fill in the cells below.

In [ ]:
# STUDENT: inspect X_swapped and write your diagnosis
# 1. What is the shape of X_swapped?
# print(f"X_swapped.shape = {X_swapped.shape}")

# 2. Which axis holds I/Q? Which holds time?
# YOUR ANSWER:

# 3. Why is this wrong for the IQ contract?
# YOUR ANSWER:

In [ ]:
# STUDENT: write the correction
# X_fixed = ...

# Verify your fix
# assert X_fixed.shape == X.shape, f"Shape mismatch: {X_fixed.shape} != {X.shape}"
# assert X_fixed.dtype == X.dtype, f"Dtype mismatch: {X_fixed.dtype} != {X.dtype}"
# assert np.array_equal(X_fixed, X), "Values do not match"
# print("AXIS SWAP CHECK: PASS")

#### Automatic Validation — PC-2

In [ ]:
# Run this cell after you have defined X_fixed above.
# If you have not defined X_fixed, uncomment the lines and complete the exercise.

# STUDENT: uncomment and fill in X_fixed before running
# X_fixed = ...  # your correction here

# --- automatic validation (do not modify below this line) ---
try:
    axis_swap_corrected = (
        X_fixed.shape == X.shape
        and X_fixed.dtype == X.dtype
        and np.array_equal(X_fixed, X)
    )
    assert X_fixed.shape == X.shape
    assert X_fixed.dtype == X.dtype
    assert np.array_equal(X_fixed, X)
    print("AXIS SWAP CHECK: PASS")
except NameError:
    axis_swap_corrected = False
    print("AXIS SWAP CHECK: WAIT — define X_fixed first")
except AssertionError as e:
    axis_swap_corrected = False
    print(f"AXIS SWAP CHECK: WAIT — {e}")

---
### PC-3 — IQ Power Agreement

Two independent methods compute the average power of the IQ signal. They must agree within tight tolerances.

**Method A** — Direct I² + Q²:
$$P_{\text{IQ}} = \text{mean}(I^2 + Q^2)$$

**Method B** — Complex magnitude:
$$z = I + jQ, \quad P_{\text{complex}} = \text{mean}(|z|^2)$$

In [ ]:
# Extract I and Q from the original (non-swapped) data
I = X[:, 0, :]  # shape (N, L)
Q = X[:, 1, :]  # shape (N, L)

print(f"I.shape = {I.shape}")
print(f"Q.shape = {Q.shape}")

In [ ]:
# Method A: direct I² + Q²
P_iq = np.mean(I**2 + Q**2)
print(f"Method A — P_iq     = {P_iq:.10f}")

In [ ]:
# Method B: complex magnitude
z = I + 1j * Q
P_complex = np.mean(np.abs(z)**2)
print(f"Method B — P_complex = {P_complex:.10f}")

In [ ]:
# Validate agreement
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, f"Power mismatch: P_iq={P_iq}, P_complex={P_complex}"

print(f"P_iq      = {P_iq:.10f}")
print(f"P_complex = {P_complex:.10f}")
print(f"Difference = {abs(P_iq - P_complex):.2e}")
print(f"Tolerance: rtol={POWER_RTOL}, atol={POWER_ATOL}")
print("POWER CHECK: PASS")

---
## PASS CRITERION GATE

In [ ]:
automatic_checks = {
    "axis_swap_corrected": axis_swap_corrected,
    "power_consistency": power_consistency,
}
automatic_pass = all(automatic_checks.values())

final_pass = automatic_pass and AXES_EXPLANATION_VERIFIED

print("=" * 50)
print("PRE-M0.3 — PASS CRITERION GATE")
print("=" * 50)
for name, passed in automatic_checks.items():
    status = "PASS" if passed else "WAIT"
    print(f"  {name:30s}: {status}")
print(f"  {'AXES_EXPLANATION_VERIFIED':30s}: {'PASS' if AXES_EXPLANATION_VERIFIED else 'WAIT'}")
print("-" * 50)

if final_pass:
    print("PRE-M0.3 FINAL STATUS: PASS")
else:
    print("PRE-M0.3 FINAL STATUS: WAIT")
print("=" * 50)

---
## OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

**Solutions are provided below. Do not look at these until you have attempted all exercises and challenges.**

### Solution — Exercise 1 (Select example 5)

In [ ]:
exercise_1 = X[5]
print(f"exercise_1.shape = {exercise_1.shape}")  # (2, 200)

### Solution — Exercise 2 (Select I across all examples)

In [ ]:
exercise_2 = X[:, 0, :]
print(f"exercise_2.shape = {exercise_2.shape}")  # (50, 200)

### Solution — Exercise 3 (Select Q of example 10)

In [ ]:
exercise_3 = X[10, 1, :]
print(f"exercise_3.shape = {exercise_3.shape}")  # (200,)

### Solution — Exercise 4 (Extract samples 20:40)

In [ ]:
exercise_4 = X[3, :, 20:40]
print(f"exercise_4.shape = {exercise_4.shape}")  # (2, 20)

### Solution — Exercise 5 (Preserve batch dimension)

In [ ]:
exercise_5 = X[7:8]
print(f"exercise_5.shape = {exercise_5.shape}")  # (1, 2, 200)

### Solution — Exercise 6 (Predict shape)

In [ ]:
# Predicted: (3, 50) — 3 examples, I component, 50 time samples
exercise_6 = X[1:4, 0, 50:100]
print(f"exercise_6.shape = {exercise_6.shape}")  # (3, 50)

### Solution — PC-2 Axis Swap Fix

**Diagnosis:**
- `X_swapped.shape = (50, 200, 2)` — axes 1 and 2 are swapped.
- Axis 1 now holds time samples (size 200) instead of I/Q (size 2).
- Axis 2 now holds I/Q (size 2) instead of time samples (size 200).
- This violates the IQ contract `X.shape == (N, 2, L)`.

**Correction:** Apply the inverse transpose to restore the original axis order.

In [ ]:
X_fixed = np.transpose(X_swapped, (0, 2, 1))
print(f"X_fixed.shape = {X_fixed.shape}")  # (50, 2, 200)
assert np.array_equal(X_fixed, X), "Recovery failed"
print("Recovery verified: X_fixed == X")